#  Notebook 1: SQL-Powered EDA — Olist Brazilian E-commerce

*100k+ real orders · SQLite · Plotly dark theme*

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text
import os, warnings, json
warnings.filterwarnings('ignore')

DB_PATH = os.path.join('..', 'data', 'olist.db')
engine = create_engine(f'sqlite:///{DB_PATH}')


## 1. Dataset Overview

In [2]:
tables = ['customers','geolocation','order_items','order_payments',
          'order_reviews','orders','products','sellers','category_translation']
print(f"{'Table':<35} {'Rows':>10}")
print("-" * 47)
for t in tables:
    with engine.connect() as conn:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
    print(f"  {t:<33} {n:>10,}")


Table                                     Rows
-----------------------------------------------
  customers                             99,441
  geolocation                        1,000,163
  order_items                          112,650
  order_payments                       103,886
  order_reviews                         99,224
  orders                                99,441
  products                              32,951
  sellers                                3,095
  category_translation                      71


## 2. Revenue by State

In [3]:
query = '''
SELECT
    c.customer_state,
    ROUND(SUM(op.payment_value), 2)      AS total_revenue,
    COUNT(DISTINCT o.order_id)           AS total_orders,
    ROUND(AVG(op.payment_value), 2)      AS avg_order_value,
    COUNT(DISTINCT o.customer_id)        AS unique_customers
FROM orders o
JOIN customers c         ON o.customer_id = c.customer_id
JOIN order_payments op   ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_revenue DESC
'''
revenue_by_state = pd.read_sql(query, engine)
print(revenue_by_state.head(10).to_string(index=False))

fig = px.bar(revenue_by_state.head(15), x='customer_state', y='total_revenue',
             color='avg_order_value', color_continuous_scale='Viridis',
             title=' Total Revenue by Brazilian State (Top 15)',
             labels={'customer_state':'State','total_revenue':'Revenue (BRL)',
                     'avg_order_value':'Avg Order Value'})
fig.update_layout(template='plotly_dark', font_family='Inter')
fig.show()


customer_state  total_revenue  total_orders  avg_order_value  unique_customers
            SP     5770266.19         40500           136.39             40500
            RJ     2055690.45         12350           158.08             12350
            MG     1819277.61         11354           154.12             11354
            RS      861802.40          5345           155.45              5345
            PR      781919.55          4923           152.45              4923
            SC      595208.40          3546           162.58              3546
            BA      591270.60          3256           169.76              3256
            DF      346146.17          2080           161.60              2080
            GO      334294.22          1957           163.31              1957
            ES      317682.65          1995           153.62              1995


## 3. Delivery Performance Analysis

In [4]:
delivery_query = '''
SELECT
    order_id,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    JULIANDAY(order_delivered_customer_date) - JULIANDAY(order_estimated_delivery_date) AS delay_days,
    CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date THEN 1 ELSE 0 END AS on_time
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
'''
delivery_df = pd.read_sql(delivery_query, engine)
print(f"Delivered orders : {len(delivery_df):,}")
print(f"On-time rate     : {delivery_df['on_time'].mean()*100:.1f}%")
print(f"Avg delay (days) : {delivery_df['delay_days'].mean():.2f}")
print(f"Max delay (days) : {delivery_df['delay_days'].max():.1f}")

fig2 = px.histogram(delivery_df, x='delay_days', nbins=80,
                    title=' Delivery Delay Distribution (days vs estimate)',
                    color_discrete_sequence=['#06b6d4'],
                    labels={'delay_days':'Delay (days)'})
fig2.add_vline(x=0, line_dash='dash', line_color='#ef4444',
               annotation_text='On-time threshold')
fig2.update_layout(template='plotly_dark', font_family='Inter')
fig2.show()


Delivered orders : 96,470
On-time rate     : 91.9%
Avg delay (days) : -11.18
Max delay (days) : 189.0


## 4. Monthly Order Volume & Revenue Trend

In [5]:
time_query = '''
SELECT
    STRFTIME('%Y-%m', order_purchase_timestamp) AS month,
    COUNT(*)                                     AS num_orders,
    ROUND(SUM(payment_value), 2)                 AS monthly_revenue
FROM orders o
JOIN order_payments op ON o.order_id = op.order_id
WHERE order_status NOT IN ('canceled','unavailable')
  AND order_purchase_timestamp >= '2017-01-01'
GROUP BY month ORDER BY month
'''
time_df = pd.read_sql(time_query, engine)

fig3 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=['Monthly Orders','Monthly Revenue (BRL)'],
                     row_heights=[0.4,0.6])
fig3.add_trace(go.Scatter(x=time_df['month'], y=time_df['num_orders'],
                          fill='tozeroy', line_color='#8b5cf6', name='Orders'), row=1, col=1)
fig3.add_trace(go.Scatter(x=time_df['month'], y=time_df['monthly_revenue'],
                          fill='tozeroy', line_color='#06b6d4', name='Revenue'), row=2, col=1)
fig3.update_layout(template='plotly_dark', font_family='Inter',
                   title=' Monthly Order Volume & Revenue Trend', height=500)
fig3.show()


## 5. Review Score vs Delivery Delay

In [6]:
review_query = '''
SELECT
    r.review_score,
    COUNT(*)  AS count,
    ROUND(AVG(JULIANDAY(o.order_delivered_customer_date) -
              JULIANDAY(o.order_estimated_delivery_date)), 2) AS avg_delay
FROM order_reviews r
JOIN orders o ON r.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY r.review_score ORDER BY r.review_score
'''
review_df = pd.read_sql(review_query, engine)
print(review_df.to_string(index=False))

fig4 = px.bar(review_df, x='review_score', y='count', color='avg_delay',
              color_continuous_scale='RdYlGn_r',
              title=' Review Score Distribution (coloured by Avg Delivery Delay)',
              labels={'review_score':'Review Score (1-5)','count':'# Reviews',
                      'avg_delay':'Avg Delay (days)'})
fig4.update_layout(template='plotly_dark', font_family='Inter')
fig4.show()


 review_score  count  avg_delay
            1   9406      -3.36
            2   2941      -7.94
            3   7961     -10.08
            4  18987     -11.68
            5  57066     -12.69


## 6. Top Product Categories by Revenue

In [7]:
cat_query = '''
SELECT
    COALESCE(ct.product_category_name_english, p.product_category_name) AS category,
    ROUND(SUM(oi.price), 2)              AS total_revenue,
    COUNT(DISTINCT oi.order_id)          AS order_count
FROM order_items oi
JOIN products p                ON oi.product_id = p.product_id
LEFT JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY category ORDER BY total_revenue DESC LIMIT 20
'''
cat_df = pd.read_sql(cat_query, engine)

fig5 = px.treemap(cat_df, path=['category'], values='total_revenue',
                  color='order_count', color_continuous_scale='Purples',
                  title=' Revenue Treemap by Product Category')
fig5.update_layout(template='plotly_dark', font_family='Inter')
fig5.show()


## 7. Export EDA Summary

In [8]:
os.makedirs('../outputs', exist_ok=True)
summary = {
    'total_orders':          int(pd.read_sql('SELECT COUNT(DISTINCT order_id) FROM orders', engine).iloc[0,0]),
    'total_customers':       int(pd.read_sql('SELECT COUNT(DISTINCT customer_id) FROM customers', engine).iloc[0,0]),
    'total_revenue_brl':     float(pd.read_sql('SELECT SUM(payment_value) FROM order_payments', engine).iloc[0,0]),
    'on_time_delivery_rate': float(delivery_df['on_time'].mean()),
    'avg_review_score':      float(pd.read_sql('SELECT AVG(review_score) FROM order_reviews', engine).iloc[0,0]),
    'avg_delay_days':        float(delivery_df['delay_days'].mean()),
}
with open('../outputs/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
for k, v in summary.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v:,}")


  total_orders: 99,441
  total_customers: 99,441
  total_revenue_brl: 16008872.1200
  on_time_delivery_rate: 0.9189
  avg_review_score: 4.0864
  avg_delay_days: -11.1781
